# 04 - Ejercicios Practicos: Segmentacion con K-Means

## Curso de Machine Learning: Modelos No Supervisados

En este cuaderno pondras en practica todos los conceptos estudiados sobre **Agrupamiento con K-Means**, seleccion de hiperparametros mediante metricas internas y perfilamiento de arquetipos. 

Para ello trabajaremos sobre un caso de estudio de gran utilidad en **Economia Aplicada, Politica Publica yRiesgo Soberano**: la **Segmentacion Macroeconomica, Energetica y Demografica de Naciones (167 paises)** para la asignacion estrategica de fondos de inversion e infraestructura internacional.

> **Nota:** Encontraras las soluciones completas y comentadas a estos ejercicios en la carpeta `.soluciones/04_Ejercicios_Practicos_Solucion.ipynb`. Te recomendamos resolver cada bloque por tu cuenta antes de revisar la solucion.

---

### Instrucciones Generales
1. Ejecuta la celda de configuracion inicial y generacion del dataset macroeconomico.
2. Lee cuidadosamente el contexto y los requerimientos de cada ejercicio.
3. Escribe tu codigo en las celdas provistas (`# TU CODIGO AQUI`).
4. Responde las preguntas de analisis en las celdas de Markdown correspondientes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## Contexto del Dataset: Indicadores de Desarrollo Global y Transicion Energetica (167 Paises)

Una agencia multilateral de inversion y desarrollo sostenible necesita clasificar **167 naciones** en grupos estrategicos homogeneos utilizando 9 indicadores cuantitativos clave:
- `mortalidad_infantil`: Muertes de menores de 5 anos por cada 1,000 nacidos vivos.
- `exportaciones_pct_pib`: Exportaciones de bienes y servicios (% del PIB).
- `gasto_salud_pct_pib`: Gasto total en salud (% del PIB).
- `importaciones_pct_pib`: Importaciones de bienes y servicios (% del PIB).
- `ingreso_per_capita_usd`: Ingreso neto medio por persona ajustado por paridad de poder adquisitivo (USD).
- `inflacion_anual_pct`: Tasa de crecimiento anual del deflactor del PIB (%).
- `esperanza_vida_anos`: Numero promedio de anos que viviria un recien nacido.
- `tasa_fertilidad`: Numero de hijos promedio por mujer.
- `emisiones_co2_per_capita`: Toneladas metricas de $CO_2$ emitidas por habitante al ano.


In [ ]:
# Generacion reproducible del dataset realista de 167 naciones (calibrado sobre estadisticas del Banco Mundial)
def cargar_dataset_macroeconomico(seed=42):
    rng = np.random.default_rng(seed)
    
    # Arquetipo 1: Economias vulnerables en vias de desarrollo (n=48)
    g1 = np.column_stack([
        rng.normal(88.0, 22.0, 48).clip(35, 160),    # mortalidad_infantil
        rng.normal(28.0, 10.0, 48).clip(10, 60),     # exportaciones_pct_pib
        rng.normal(5.8, 1.8, 48).clip(2.5, 11.0),    # gasto_salud_pct_pib
        rng.normal(42.0, 12.0, 48).clip(20, 85),     # importaciones_pct_pib
        rng.lognormal(7.6, 0.45, 48).clip(700, 4500),# ingreso_per_capita_usd
        rng.normal(11.5, 6.0, 48).clip(1.0, 35.0),   # inflacion_anual_pct
        rng.normal(59.5, 4.5, 48).clip(48, 68),      # esperanza_vida_anos
        rng.normal(4.8, 0.9, 48).clip(2.8, 7.2),     # tasa_fertilidad
        rng.normal(0.9, 0.5, 48).clip(0.1, 2.5)      # emisiones_co2_per_capita
    ])
    
    # Arquetipo 2: Economias emergentes e industriales en expansion (n=67)
    g2 = np.column_stack([
        rng.normal(21.0, 8.5, 67).clip(6, 45),       # mortalidad_infantil
        rng.normal(39.0, 13.0, 67).clip(12, 75),     # exportaciones_pct_pib
        rng.normal(6.5, 1.6, 67).clip(3.2, 10.5),    # gasto_salud_pct_pib
        rng.normal(44.0, 13.0, 67).clip(15, 80),     # importaciones_pct_pib
        rng.lognormal(9.4, 0.40, 67).clip(5000, 24000), # ingreso_per_capita_usd
        rng.normal(6.2, 3.8, 67).clip(-0.5, 20.0),   # inflacion_anual_pct
        rng.normal(73.2, 3.0, 67).clip(66, 79),      # esperanza_vida_anos
        rng.normal(2.2, 0.45, 67).clip(1.3, 3.4),    # tasa_fertilidad
        rng.normal(4.8, 1.8, 67).clip(1.5, 9.5)      # emisiones_co2_per_capita
    ])
    
    # Arquetipo 3: Economias avanzadas de altos ingresos y servicios (n=37)
    g3 = np.column_stack([
        rng.normal(4.5, 1.5, 37).clip(2.0, 9.0),     # mortalidad_infantil
        rng.normal(48.0, 16.0, 37).clip(18, 90),     # exportaciones_pct_pib
        rng.normal(9.8, 1.9, 37).clip(6.5, 15.0),    # gasto_salud_pct_pib
        rng.normal(45.0, 15.0, 37).clip(18, 85),     # importaciones_pct_pib
        rng.normal(44500, 9500, 37).clip(28000, 72000), # ingreso_per_capita_usd
        rng.normal(2.1, 1.2, 37).clip(-1.0, 5.5),    # inflacion_anual_pct
        rng.normal(81.5, 1.8, 37).clip(77, 85),      # esperanza_vida_anos
        rng.normal(1.6, 0.25, 37).clip(1.1, 2.2),    # tasa_fertilidad
        rng.normal(8.9, 2.6, 37).clip(4.0, 15.5)     # emisiones_co2_per_capita
    ])
    
    # Arquetipo 4: Hubs financieros / Exportadores energeticos intensivos (n=15)
    g4 = np.column_stack([
        rng.normal(7.5, 3.0, 15).clip(3.0, 14.0),    # mortalidad_infantil
        rng.normal(82.0, 22.0, 15).clip(55, 140),    # exportaciones_pct_pib
        rng.normal(4.5, 1.5, 15).clip(2.2, 7.8),     # gasto_salud_pct_pib
        rng.normal(68.0, 20.0, 15).clip(40, 115),    # importaciones_pct_pib
        rng.normal(68000, 14000, 15).clip(48000, 105000), # ingreso_per_capita_usd
        rng.normal(3.8, 2.5, 15).clip(0.0, 9.5),     # inflacion_anual_pct
        rng.normal(79.0, 2.2, 15).clip(75, 83),      # esperanza_vida_anos
        rng.normal(1.85, 0.35, 15).clip(1.2, 2.6),   # tasa_fertilidad
        rng.normal(21.5, 5.0, 15).clip(14.0, 32.0)   # emisiones_co2_per_capita
    ])
    
    X_mat = np.vstack([g1, g2, g3, g4])
    idx_perm = rng.permutation(X_mat.shape[0])
    X_mat = X_mat[idx_perm]
    
    columnas = [
        'mortalidad_infantil', 'exportaciones_pct_pib', 'gasto_salud_pct_pib',
        'importaciones_pct_pib', 'ingreso_per_capita_usd', 'inflacion_anual_pct',
        'esperanza_vida_anos', 'tasa_fertilidad', 'emisiones_co2_per_capita'
    ]
    codigos_pais = [f'PAIS_{i:03d}' for i in range(1, X_mat.shape[0] + 1)]
    return pd.DataFrame(X_mat, index=codigos_pais, columns=columnas)

df_paises = cargar_dataset_macroeconomico()
print(f"Dimensiones del dataset: {df_paises.shape}")
df_paises.head()


---
## Ejercicio 1: Diagnostico de Escala y Necesidad de Estandarizacion

1. Calcula e imprime las estadisticas descriptivas (`mean`, `std`, `min`, `max`) de `df_paises`.
2. Entrena un modelo `KMeans(n_clusters=4, random_state=42, n_init=10)` directamente sobre los datos crudos sin escalar (`df_paises`).
3. Estandariza los datos utilizando `StandardScaler` y guarda la matriz resultante en `X_scaled`.
4. Entrena un segundo modelo `KMeans(n_clusters=4, random_state=42, n_init=10)` sobre `X_scaled`.
5. Compara el `silhouette_score` de ambos modelos (evaluando cada uno en su respectivo espacio de entrada) y analiza la correlacion entre las etiquetas del modelo sin escalar y la variable `ingreso_per_capita_usd`.


In [ ]:
# TU CODIGO AQUI: Estadisticas, K-Means sin escalar vs K-Means estandarizado



**Tus Observaciones (Ejercicio 1):**
*Escribe aqui tus conclusiones sobre como afecta la magnitud de `ingreso_per_capita_usd` al modelo sin escalar...*


---
## Ejercicio 2: Determinacion del Numero Optimo de Clusters ($K^*$)

Utilizando exclusivamente la matriz estandarizada `X_scaled`:
1. Explora los valores de $K \in \{2, 3, 4, 5, 6, 7, 8\}$ entrenando un modelo `KMeans(n_clusters=k, init='k-means++', n_init=15, random_state=42)` para cada valor.
2. Almacena para cada $K$: la **Inercia**, el **Coeficiente de Silueta**, el **Indice de Calinski-Harabasz** y el **Indice de Davies-Bouldin**.
3. Genera una figura con subplots mostrando la evolucion de las metricas en funcion de $K$.
4. **Pregunta:** ¿Cual es el numero optimo de grupos $K^*$ donde convergen las metricas de validacion interna?


In [ ]:
# TU CODIGO AQUI: Bucle de evaluacion para K de 2 a 8 y graficos de validacion



**Tus Observaciones (Ejercicio 2):**
*Escribe aqui tu respuesta justificando el valor de $K^*$ seleccionado...*


---
## Ejercicio 3: Entrenamiento del Modelo Optimo y Reconstruccion de Centroides

1. Entrena el modelo final `kmeans_opt` con el numero optimo de clusters $K^* = 4$ sobre `X_scaled`.
2. Agrega una columna llamada `'Cluster'` al DataFrame original `df_paises` con las etiquetas predichas.
3. Extrae los centroides en el espacio estandarizado ($Z$-scores) y graficalos mediante un `sns.heatmap` centrado en 0.
4. Utiliza `scaler.inverse_transform()` sobre `kmeans_opt.cluster_centers_` para construir un DataFrame con los promedios de cada cluster en las **unidades macroeconomicas reales** (USD, anos de vida, % del PIB, etc.).


In [ ]:
# TU CODIGO AQUI: Entrenamiento con K=4, Heatmap de Z-scores y Tabla en unidades reales



---
## Ejercicio 4: Proyeccion con PCA y Visualizacion Topologica

1. Aplica `PCA(n_components=2, random_state=42)` sobre `X_scaled` para proyectar los 167 paises en un plano bidimensional.
2. Proyecta tambien los centroides del modelo (`kmeans_opt.cluster_centers_`) usando el mismo objeto `pca.transform()`.
3. Crea un grafico de dispersion (*scatter plot*) coloreando cada pais segun su cluster y dibujando los 4 centroides con un marcador destacado. Indica en los ejes el porcentaje de varianza explicada por cada componente.


In [ ]:
# TU CODIGO AQUI: Proyeccion PCA en 2D de paises y centroides



---
## Ejercicio 5: Reglas de Politica Publica mediante Arbol Sustituto

Para que el comite del Banco de Desarrollo pueda clasificar nuevos paises rapidamente mediante umbrales claros:
1. Entrena un `DecisionTreeClassifier(max_depth=3, random_state=42)` utilizando las variables en su escala original (`df_paises.drop(columns=['Cluster'])`) como predictoras y la columna `'Cluster'` como objetivo.
2. Calcula la exactitud (*fidelidad*) del arbol sobre el conjunto de datos y visualizalo con `plot_tree`.
3. Basandote en la tabla de centroides reales del Ejercicio 3 y en el arbol, redacta una breve descripcion de 1 linea bautizando cada uno de los 4 grupos socioeconomicos identificados y menciona cual cluster requiere ayuda humanitaria urgente en salud e infancia.


In [ ]:
# TU CODIGO AQUI: Arbol de decision sustituto e interpretacion final



**Tus Conclusiones y Bautizo de Clusters (Ejercicio 5):**
- **Cluster 0:** *...*
- **Cluster 1:** *...*
- **Cluster 2:** *...*
- **Cluster 3:** *...*
